# Exp15 — Do the shared IOI heads *write* the same thing across languages?

**The sentence:** shared heads (Exp13/14) tell us the *interface* is shared. This tests whether the **residual-stream write** of those heads is the same object in EN/FR/YO, or a language-specific transformation of it — no erasure, no subspace definition.

**Design (matched, same as Exp14):** same English names / order / template index across languages, so the answer token is identical. For the top-K EN-ranked heads, take each head's actual contribution to the residual stream at the END position (post-attn-norm linearised, exact when summed over a layer).

**Metrics**
1. *Name-conditioned cosine.* Per-name mean write m_L[n]. Compare cos(m_EN[n], m_L[n]) against a **floor** (same language, template-disjoint halves). Raw and name-centred (centred removes the shared offset, keeps name-dependent structure).
2. *Energy decomposition of the language shift* D_n = m_L[n] − m_EN[n]: constant offset (language tag) vs name-dependent part (language-specific format), vs split-half floor.
3. *Negative control:* permute names across languages. Centred cosine must collapse, otherwise the metric can't see name structure.

**Gates (print before trusting anything):** ranking sanity (top-K ablation << random-K ablation), floor cosine high.
Heads are ranked on a *separate* seed from the analysis seeds (held-out prompts).

In [1]:
import os
GPU_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["HF_HUB_OFFLINE"] = "1"
import json, random, gc, re
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELS = {
    "2b-base": "/home2/models/google-gemma-2-2b-base",
    "9b-base": "/home/models/gemma-2-9b",
    "9b-it":   "/home/models/gemma-2-9b-it",
    "27b-it":  "/home/models/gemma-2-27b-it",
}
RUN = ["2b-base", "9b-it", "27b-it"]      # edit
LANGS = ["en", "fr", "yo"]
N_NAMES = 20
PAIRS_PER_TEMPLATE = 60
K_HEADS = 10
RANK_SEED = 100
SEEDS = [0, 1, 2]
BATCH = 16
OUT = "exp15_results"
FORCE = False
os.makedirs(OUT, exist_ok=True)

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NAME_CANDIDATES = ["John","Mary","Tom","Alice","David","Sarah","James","Emily","Michael","Laura",
    "Peter","Anna","Mark","Julia","Paul","Susan","Henry","Kate","Robert","Linda",
    "George","Helen","Frank","Diana","Oscar","Nora","Simon","Ruth","Victor","Grace"]

TEMPL = {
 "en": ["When {X} and {Y} went to the market, {S} gave a book to",
        "After school, {X} and {Y} were at home. {S} handed a gift to",
        "One day, {X} and {Y} met in the park. {S} passed the ball to",
        "Then {X} and {Y} had lunch together, and {S} gave some money to",
        "Yesterday {X} and {Y} arrived at the shop, and {S} gave the bag to",
        "At the party, {X} and {Y} talked for a while. {S} offered a drink to"],
 "fr": ["Quand {X} et {Y} sont allés au marché, {S} a donné un livre à",
        "Après l'école, {X} et {Y} étaient à la maison. {S} a remis un cadeau à",
        "Un jour, {X} et {Y} se sont retrouvés au parc. {S} a passé le ballon à",
        "Ensuite {X} et {Y} ont déjeuné ensemble, et {S} a donné de l'argent à",
        "Hier {X} et {Y} sont arrivés au magasin, et {S} a donné le sac à",
        "À la fête, {X} et {Y} ont parlé un moment. {S} a offert un verre à"],
 # NOTE: Yoruba templates are mine — have a Yoruba speaker sanity-check them.
 "yo": ["Nígbà tí {X} àti {Y} lọ sí ọjà, {S} fi ìwé fún",
        "Lẹ́yìn ilé ìwé, {X} àti {Y} wà ní ilé. {S} fi ẹ̀bùn fún",
        "Ní ọjọ́ kan, {X} àti {Y} pàdé ní ọgbà. {S} fi bọ́ọ̀lù fún",
        "Lẹ́yìn náà {X} àti {Y} jẹun papọ̀, {S} sì fi owó fún",
        "Lánàá {X} àti {Y} dé ilé ìtajà, {S} sì fi àpò fún",
        "Ní ibi àríyá, {X} àti {Y} sọ̀rọ̀ fún ìgbà díẹ̀. {S} fi ohun mímu fún"],
}
N_T = 6

def build_pairs(names, seed):
    rng = random.Random(seed)
    out = []
    for t in range(N_T):
        out.append([(*rng.sample(names, 2), rng.random() < 0.5) for _ in range(PAIRS_PER_TEMPLATE)])
    return out   # out[t] = [(io, s, io_first)]

def make_prompts(lang, pairs):
    prompts, meta = [], []
    for t, lst in enumerate(pairs):
        for io, s, f in lst:
            X, Y = (io, s) if f else (s, io)
            prompts.append(TEMPL[lang][t].format(X=X, Y=Y, S=s))
            meta.append((t, io, s))
    return prompts, meta

In [3]:
def load(path):
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16,
                                                 device_map={"": 0}, attn_implementation="eager").eval()
    return tok, model

def dims(model):
    c = model.config
    return c.num_hidden_layers, c.num_attention_heads, c.head_dim, c.hidden_size

def single_tok(tok, name):
    ids = tok.encode(" " + name, add_special_tokens=False)
    return ids[0] if len(ids) == 1 else None

@torch.no_grad()
def dla_scores(model, tok, prompts, io, s):
    """Linearised direct-logit-attribution of every head to (logit[IO]-logit[S]) at END. -> (n, L, H)"""
    NL, H, hd, d = dims(model); dev = next(model.parameters()).device
    eps = model.config.rms_norm_eps
    WU = model.lm_head.weight
    wf = 1 + model.model.norm.weight.float()
    ctx, hs, res = {}, [], []
    for l, layer in enumerate(model.model.layers):
        wpost = 1 + layer.post_attention_layernorm.weight.float()
        def hook(mod, args, out, l=l, wpost=wpost):
            z = args[0][:, -1, :].float(); B = z.shape[0]; z = z.view(B, H, hd)
            rms = ((out[:, -1, :].float() ** 2).mean(-1) + eps).sqrt()
            Wo = mod.weight.float().view(d, H, hd)
            v = torch.einsum("dhe,bd->bhe", Wo, wpost[None] * ctx["uf"])
            ctx["dots"][:, l, :] = (z * v).sum(-1) / rms[:, None]
        hs.append(layer.self_attn.o_proj.register_forward_hook(hook))
    def fpre(mod, args): ctx["rmsf"] = ((args[0][:, -1, :].float() ** 2).mean(-1) + eps).sqrt()
    hs.append(model.model.norm.register_forward_pre_hook(fpre))
    try:
        for i in range(0, len(prompts), BATCH):
            enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True).to(dev)
            io_b = torch.tensor(io[i:i+BATCH], device=dev); s_b = torch.tensor(s[i:i+BATCH], device=dev)
            ctx["uf"] = wf[None] * (WU[io_b] - WU[s_b]).float()
            ctx["dots"] = torch.zeros(len(io_b), NL, H, device=dev)
            model.model(**enc)
            res.append((ctx["dots"] / ctx["rmsf"][:, None, None]).cpu())
    finally:
        for h in hs: h.remove()
    return torch.cat(res).numpy()

@torch.no_grad()
def logit_diff(model, tok, prompts, io, s, ablate=None):
    """mean-able per-prompt logit[IO]-logit[S]; ablate = {layer: [heads]} zero-ablates z at all positions."""
    NL, H, hd, d = dims(model); dev = next(model.parameters()).device
    cap = getattr(model.config, "final_logit_softcapping", None)
    hs, out = [], []
    for l, hl in (ablate or {}).items():
        def pre(mod, args, hl=hl):
            x = args[0].clone(); B, T, _ = x.shape
            x = x.view(B, T, H, hd); x[:, :, hl, :] = 0
            return (x.view(B, T, H * hd),) + tuple(args[1:])
        hs.append(model.model.layers[l].self_attn.o_proj.register_forward_pre_hook(pre))
    try:
        for i in range(0, len(prompts), BATCH):
            enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True).to(dev)
            h = model.model(**enc).last_hidden_state[:, -1]
            lg = model.lm_head(h).float()
            if cap: lg = torch.tanh(lg / cap) * cap
            io_b = torch.tensor(io[i:i+BATCH], device=dev); s_b = torch.tensor(s[i:i+BATCH], device=dev)
            ar = torch.arange(len(io_b), device=dev)
            out.append((lg[ar, io_b] - lg[ar, s_b]).cpu())
    finally:
        for h in hs: h.remove()
    return torch.cat(out).numpy()

@torch.no_grad()
def collect_heads(model, tok, prompts, heads):
    """Actual residual-stream write of each listed head at END (post-attn-norm linearised). -> (n, K, d)"""
    NL, H, hd, d = dims(model); dev = next(model.parameters()).device
    eps = model.config.rms_norm_eps
    K = len(heads); by_layer = {}
    for k, (l, h) in enumerate(heads): by_layer.setdefault(l, []).append((k, h))
    ctx, hs = {}, []
    for l, kh in by_layer.items():
        ks = [k for k, _ in kh]; hh = [h for _, h in kh]
        wpost = 1 + model.model.layers[l].post_attention_layernorm.weight.float()
        def hook(mod, args, out, ks=ks, hh=hh, wpost=wpost):
            z = args[0][:, -1, :].float(); B = z.shape[0]; z = z.view(B, H, hd)[:, hh]
            rms = ((out[:, -1, :].float() ** 2).mean(-1) + eps).sqrt()
            Wo = mod.weight.float().view(d, H, hd)[:, hh]
            c = torch.einsum("bhe,dhe->bhd", z, Wo) * wpost[None, None] / rms[:, None, None]
            ctx["buf"][:, ks, :] = c
        hs.append(model.model.layers[l].self_attn.o_proj.register_forward_hook(hook))
    res = []
    try:
        for i in range(0, len(prompts), BATCH):
            enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True).to(dev)
            ctx["buf"] = torch.zeros(enc["input_ids"].shape[0], K, d, device=dev)
            model.model(**enc)
            res.append(ctx["buf"].cpu())
    finally:
        for h in hs: h.remove()
    return torch.cat(res).numpy()

In [4]:
def name_means(X, nidx, mask, N):
    return np.stack([X[(nidx == n) & mask].mean(0) for n in range(N)])

def cosrows(A, B):
    return (A * B).sum(1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12)

def metrics(X_en, X_l, nidx, t, N, rng):
    """X_*: (n, d) per-prompt writes (aligned by construction). Returns dict of scalars."""
    h1, h2, ha = t < 3, t >= 3, np.ones_like(t, bool)
    en1, en2 = name_means(X_en, nidx, h1, N), name_means(X_en, nidx, h2, N)
    l1, l2 = name_means(X_l, nidx, h1, N), name_means(X_l, nidx, h2, N)
    r = {}
    shift = int(rng.integers(1, N)); perm = (np.arange(N) + shift) % N
    for tag, cen in (("raw", False), ("cen", True)):
        f = (lambda a: a - a.mean(0, keepdims=True)) if cen else (lambda a: a)
        a1, a2, b1, b2 = map(f, (en1, en2, l1, l2))
        floor = 0.5 * (cosrows(a1, a2).mean() + cosrows(b1, b2).mean())
        cross = 0.5 * (cosrows(a1, b2).mean() + cosrows(b1, a2).mean())
        cperm = 0.5 * (cosrows(a1, b2[perm]).mean() + cosrows(b1[perm], a2).mean())
        r.update({f"floor_{tag}": floor, f"cross_{tag}": cross, f"perm_{tag}": cperm, f"gap_{tag}": floor - cross})
    ea, la = name_means(X_en, nidx, ha, N), name_means(X_l, nidx, ha, N)
    D = la - ea; Dm = D.mean(0, keepdims=True)
    E_tot = (ea ** 2).sum(1).mean()
    D1, D2 = l1 - en1, l2 - en2
    r["E_off"] = float((Dm ** 2).sum() / E_tot)
    r["E_dep"] = float(((D - Dm) ** 2).sum(1).mean() / E_tot)
    r["E_floor"] = float(((D1 - D2) ** 2).sum(1).mean() / 4 / E_tot)
    return {k: float(v) for k, v in r.items()}

In [5]:
def run_model(key):
    path = MODELS[key]
    tok, model = load(path)
    NL, H, hd, d = dims(model)
    names = [n for n in NAME_CANDIDATES if single_tok(tok, n) is not None][:N_NAMES]
    assert len(names) == N_NAMES, f"only {len(names)} single-token names"
    tid = {n: single_tok(tok, n) for n in names}
    nid = {n: i for i, n in enumerate(names)}

    def data(lang, seed):
        pairs = build_pairs(names, seed)
        P, M = make_prompts(lang, pairs)
        t = np.array([m[0] for m in M]); nidx = np.array([nid[m[1]] for m in M])
        io = [tid[m[1]] for m in M]; s = [tid[m[2]] for m in M]
        return P, io, s, t, nidx

    # ---- rank heads on EN, held-out seed
    P, io, s, _, _ = data("en", RANK_SEED)
    mean_dla = dla_scores(model, tok, P, io, s).mean(0)
    top = np.argsort(mean_dla.ravel())[::-1][:K_HEADS]
    heads = [(int(i // H), int(i % H)) for i in top]
    rest = [i for i in np.argsort(mean_dla.ravel())[::-1][K_HEADS:]]
    print(key, "top heads (L,H,meanDLA):", [(l, h, round(float(mean_dla[l, h]), 2)) for l, h in heads])

    # ---- gate 1: ranking sanity (top-K ablation vs random-K), all languages, seed 0
    gate = {}
    rng = np.random.default_rng(0)
    def abl(hlist):
        a = {}
        for l, h in hlist: a.setdefault(l, []).append(h)
        return a
    rand_sets = [[(int(i // H), int(i % H)) for i in rng.choice(rest, K_HEADS, replace=False)] for _ in range(3)]
    for lang in LANGS:
        P, io, s, _, _ = data(lang, 0)
        clean = logit_diff(model, tok, P, io, s).mean()
        top_r = logit_diff(model, tok, P, io, s, abl(heads)).mean() / clean
        rnd_r = np.mean([logit_diff(model, tok, P, io, s, abl(rs)).mean() / clean for rs in rand_sets])
        gate[lang] = dict(clean_ld=float(clean), retained_top=float(top_r), retained_rand=float(rnd_r))
    print("GATE ranking (retained logit-diff; top should be << random):", gate)

    # ---- main: per-seed collect + metrics
    per_seed = []
    for seed in SEEDS:
        X, nidx, t = {}, None, None
        for lang in LANGS:
            P, io, s, t_, n_ = data(lang, seed)
            X[lang] = collect_heads(model, tok, P, heads)
            t, nidx = t_, n_
        rng = np.random.default_rng(1000 + seed)
        rec = {}
        for lang in ["fr", "yo"]:
            rec[lang] = {"total": metrics(X["en"].sum(1), X[lang].sum(1), nidx, t, N_NAMES, rng),
                         "heads": [metrics(X["en"][:, k], X[lang][:, k], nidx, t, N_NAMES, rng) for k in range(K_HEADS)]}
        per_seed.append(rec)
        print(f"  seed {seed} done")
    res = dict(model=key, heads=heads, mean_dla=[float(mean_dla[l, h]) for l, h in heads], gate=gate, per_seed=per_seed)
    json.dump(res, open(f"{OUT}/{key}.json", "w"))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

for key in RUN:
    if os.path.exists(f"{OUT}/{key}.json") and not FORCE:
        print(key, "cached"); continue
    run_model(key)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/home2/models/google-gemma-2-2b-base'. Use `repo_type` argument if needed.

In [ ]:
print(type(MODELS)); print(MODELS)

<class 'list'>
['/home/models/gemma-2-9b-it', '/home/models/gemma-2-27b-it', 'google/gemma-2-2b', '/home/models/gemma-2-9b']


In [ ]:
# ---- summary
def agg(vals): return f"{np.mean(vals):+.3f}±{np.std(vals):.3f}"
rows = []
for key in RUN:
    p = f"{OUT}/{key}.json"
    if not os.path.exists(p): continue
    R = json.load(open(p))
    for lang in ["fr", "yo"]:
        g = lambda f, who="total": [s[lang]["total"][f] for s in R["per_seed"]]
        rows.append(dict(model=key, lang=lang,
            floor_raw=agg(g("floor_raw")), cross_raw=agg(g("cross_raw")), gap_raw=agg(g("gap_raw")),
            floor_cen=agg(g("floor_cen")), cross_cen=agg(g("cross_cen")), perm_cen=agg(g("perm_cen")), gap_cen=agg(g("gap_cen")),
            E_off=agg(g("E_off")), E_dep=agg(g("E_dep")), E_floor=agg(g("E_floor"))))
df = pd.DataFrame(rows); print("TOTAL WRITE (sum of top-K heads), mean±sd over seeds"); display(df)

hrows = []
for key in RUN:
    p = f"{OUT}/{key}.json"
    if not os.path.exists(p): continue
    R = json.load(open(p))
    for lang in ["fr", "yo"]:
        for k, (l, h) in enumerate(R["heads"]):
            hrows.append(dict(model=key, lang=lang, head=f"L{l}H{h}",
                floor_cen=np.mean([s[lang]["heads"][k]["floor_cen"] for s in R["per_seed"]]),
                gap_cen=np.mean([s[lang]["heads"][k]["gap_cen"] for s in R["per_seed"]]),
                gap_raw=np.mean([s[lang]["heads"][k]["gap_raw"] for s in R["per_seed"]])))
hdf = pd.DataFrame(hrows); print("PER HEAD"); display(hdf.round(3))

## Reading the output
- **Gate 0 (ranking):** retained_top must be clearly below retained_rand in every language, else the head list is junk — stop.
- **Gate 1 (floor):** `floor_cen` (same-language, template-disjoint) must be high (≳0.8). If not, the metric can't resolve anything at that scale — report as uninformative, not as "shared".
- **Gate 2 (metric range):** `perm_cen` should be ≈0 or negative while `floor_cen` is high. If perm_cen ≈ floor_cen, name structure isn't what is being measured.
- **Shared down to the write:** `gap_cen` ≈ 0 and `E_dep` ≲ `E_floor`; `E_off` may be non-zero (a constant language offset = tag, not a different format).
- **Language-specific format on shared heads:** `gap_cen` clearly > 0 *with floor intact*, and `E_dep` ≫ `E_floor`. This is the "shared interface, private internals" signature, now at the write.
- Per-head table: check whether a gap is concentrated in a few heads (e.g. the heads Yoruba-specific ablation flagged) or spread out.
- Caveat to keep in the paper: names are English in all languages (matched design), so a gap cannot be a name-lexical artefact, but it *can* reflect context-dependent attention/value changes caused by the surrounding language. That is the effect, not a confound.